# 03 — Model Training

This notebook trains and evaluates the classical ML models:
- **Logistic Regression** (simple baseline)
- **XGBoost** (strong ML baseline)

All models are trained on **all feature sets** directly in the loop — no separate showcase model.

Results are saved to `outputs/results/metrics.csv`.  
Plots are saved to `outputs/figures/models/`.

## 0 · Imports

In [1]:
import sys
sys.path.append('..')

from sklearn.model_selection import train_test_split

'''Self modules'''
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y, train_logistic_regression, train_xgboost
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from config import FEATURE_SETS, FEATURE_CONFIG

## 1 · Load & Split

Split is done on the **raw** DataFrame before preprocessing.  
This prevents any data leakage from imputation or encoding statistics.

In [2]:
df_raw = load_data()

df_train_raw, df_test_raw = split_data(df_raw, test_seasons=[2022, 2023])

Cache found - loading data from: C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\data\cache\pbp_raw.parquet
[split_data] Train seasons : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021)]
[split_data] Test  seasons : [np.int32(2022), np.int32(2023)]
[split_data] Train rows    : 286,892
[split_data] Test  rows    : 99,099


## 2 · Feature Set Loop

Preprocess the largest feature set once, then slice subsets from it to avoid redundant work.

Both models are trained on **all feature sets** and results are collected.
Results are appended to `metrics.csv` — existing rows for the same
`(model, feature_set)` combination are automatically overwritten.

In [3]:
# preprocess the largest feature set once; reuse column subsets to avoid redundant work.
LARGEST_FS = max(FEATURE_SETS, key=lambda k: len(FEATURE_SETS[k]))
_df_train_full = preprocess(df_train_raw, feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)
_df_test_full  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)

# align test on train columns (missing = 0, extra = dropped)
_df_test_full = _df_test_full.reindex(columns=_df_train_full.columns, fill_value=0)


                              PREPROCESSING REPORT                              

INPUT:
   Shape: (286892, 372)
   Missing: 38714249 NaNs (36.28%)

OUTPUT:
   Shape: (203362, 122)
   Features (X): 121 columns
   Target (y): 203362 samples
   Missing: 0 NaNs

SELECTED FEATURES IMPUTATION DETAILS:
   Feature Name                   | NaNs Before  | NaNs After  | Filled/Removed
   ----------------------------------------------------------------------
   down                           | 743          | 0           | 743      | dropped (<1% NaNs)      
   ydstogo                        | 0            | 0           | 0        | -                       
   yardline_100                   | 0            | 0           | 0        | -                       
   goal_to_go                     | 0            | 0           | 0        | -                       
   shotgun                        | 0            | 0           | 0        | -                       
   no_huddle                      | 0     

In [4]:
# update feature sets with encoded column names before the loop
FEATURE_SETS_ENCODED = {
    fs_name: [
        c for c in _df_train_full.columns
        if c != "target" and any(c == f or c.startswith(f + "_") for f in fs_cols)
    ]
    for fs_name, fs_cols in FEATURE_SETS.items()
}
FEATURE_SETS_ENCODED

{'mini': ['down',
  'ydstogo',
  'shotgun',
  'no_huddle',
  'score_differential',
  'posteam_timeouts_remaining',
  'quarter_seconds_remaining'],
 'comprehensive': ['down',
  'ydstogo',
  'yardline_100',
  'goal_to_go',
  'shotgun',
  'no_huddle',
  'score_differential',
  'defteam_score',
  'posteam_timeouts_remaining',
  'defteam_timeouts_remaining',
  'game_seconds_remaining',
  'half_seconds_remaining',
  'qtr',
  'ep',
  'wp',
  'total_line',
  'roof_dome',
  'roof_open',
  'roof_outdoors',
  'drive_start_transition_BLOCKED_FG,_DOWNS',
  'drive_start_transition_BLOCKED_PUNT',
  'drive_start_transition_BLOCKED_PUNT,_DOWNS',
  'drive_start_transition_DOWNS',
  'drive_start_transition_FUMBLE',
  'drive_start_transition_INTERCEPTION',
  'drive_start_transition_KICKOFF',
  'drive_start_transition_MISSED_FG',
  'drive_start_transition_MUFFED_KICKOFF',
  'drive_start_transition_MUFFED_PUNT',
  'drive_start_transition_ONSIDE_KICK',
  'drive_start_transition_OWN_KICKOFF',
  'drive_start_t

In [5]:
for fs_name, fs_cols in FEATURE_SETS_ENCODED.items():

    print(f"\n{'='*55}")
    print(f" Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    # slice subset from the preprocessed full df
    is_subset = all(c in _df_train_full.columns for c in fs_cols)
    if is_subset:
        _df_train = _df_train_full[fs_cols + ["target"]]
        _df_test  = _df_test_full[fs_cols  + ["target"]]
    else:
        _df_train = preprocess(df_train_raw, feature_set=fs_cols, feature_config=FEATURE_CONFIG)
        _df_test  = preprocess(df_test_raw,  feature_set=fs_cols, feature_config=FEATURE_CONFIG)

    # seperate feature and target
    _X_train, _y_train = get_X_y(_df_train)
    _X_test,  _y_test  = get_X_y(_df_test)

    # sanity check
    print(f"Train NaNs : {_df_train.isna().sum().sum()}")
    print(f"Test  NaNs : {_df_test.isna().sum().sum()}")
    print(f"Train shape: {_df_train.shape}")
    print(f"Test  shape: {_df_test.shape}")
    print(f"Target distribution (train):\n{_df_train['target'].value_counts(normalize=True).round(3)}")

    # Logistic Regression
    _lr = train_logistic_regression(_X_train, _y_train)
    evaluate_model(_lr, _X_test, _y_test, "LogisticRegression", fs_name)
    plot_confusion_matrix(_lr, _X_test, _y_test, "LogisticRegression", fs_name)

    # XGBoost
    _xgb = train_xgboost(_X_train, _y_train)
    evaluate_model(_xgb, _X_test, _y_test, "XGBoost", fs_name)
    plot_confusion_matrix(_xgb, _X_test, _y_test, "XGBoost", fs_name)

    # plot ROC ruce for both models
    plot_roc_curve(
        models={"LogisticRegression": _lr, "XGBoost": _xgb},
        X_test=_X_test,
        y_test=_y_test,
        feature_set=fs_name,
    )

print("\nDone. All results saved to metrics.csv")


 Feature set: mini  (7 features)
Train NaNs : 0
Test  NaNs : 0
Train shape: (203362, 8)
Test  shape: (70778, 8)
Target distribution (train):
target
1    0.591
0    0.409
Name: proportion, dtype: float64
[train_logistic_regression] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] LogisticRegression | feature_set=mini
  Accuracy  : 0.7055
  F1        : 0.6991
  ROC-AUC   : 0.7574
  Precision : 0.7033
  Recall    : 0.7055

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_LogisticRegression_mini.png


c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:29:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=mini
  Accuracy  : 0.7133
  F1        : 0.7063
  ROC-AUC   : 0.7782
  Precision : 0.712
  Recall    : 0.7133

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_XGBoost_mini.png
[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_mini.png

 Feature set: comprehensive  (33 features)
Train NaNs : 0
Test  NaNs : 0
Train shape: (203362, 34)
Test  shape: (70778, 34)
Target distribution (train):
target
1    0.591
0    0.409
Name: proportion, dtype: float64
[train_logistic_regression] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] LogisticRegr

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:29:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=comprehensive
  Accuracy  : 0.7215
  F1        : 0.7175
  ROC-AUC   : 0.7919
  Precision : 0.7193
  Recall    : 0.7215

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_XGBoost_comprehensive.png
[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_comprehensive.png

 Feature set: maxi  (121 features)
Train NaNs : 0
Test  NaNs : 0
Train shape: (203362, 122)
Test  shape: (70778, 122)
Target distribution (train):
target
1    0.591
0    0.409
Name: proportion, dtype: float64
[train_logistic_regression] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evalua

c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:30:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[train_xgboost] Training complete.
[evaluate_model] Metrics saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\results\metrics.csv

[evaluate_model] XGBoost | feature_set=maxi
  Accuracy  : 0.7307
  F1        : 0.7279
  ROC-AUC   : 0.8049
  Precision : 0.7287
  Recall    : 0.7307

[plot_confusion_matrix] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\cm_XGBoost_maxi.png
[plot_roc_curve] Saved in C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\outputs\figures\models\roc_maxi.png

Done. All results saved to metrics.csv
